<div style="font-size:2.3em;font-weight:600;margin-bottom:10px">Agentic Artificial Intelligence</div>
<div style="font-size:1.7em;margin-bottom:14px">Unit 08 · Augmented generation: RAG and CAG</div>

Today we give a model knowledge that it **never read in training**.

<div style="display:grid;grid-template-columns:1fr 1fr;gap:8px 32px"><div><b>1</b>&nbsp; Opening: two ways to bring knowledge</div><div><b>6</b>&nbsp; The cheat sheet: CAG</div><div><b>2</b>&nbsp; How the librarian finds the right pages</div><div><b>7</b>&nbsp; Which one should you use?</div><div><b>3</b>&nbsp; Building the library</div><div><b>8</b>&nbsp; Better searching</div><div><b>4</b>&nbsp; The librarian and the RAG chain</div><div><b>9</b>&nbsp; Your exercises</div><div><b>5</b>&nbsp; RAG as a tool for an agent</div><div><b>10</b>&nbsp; Wrap-up</div></div>

<div style="height:44px"></div>

## Part 1 of 10 · Opening: two ways to bring knowledge

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**How can a model answer questions about texts that it never read?**

#### 1.1 · An exam, and you may bring help

**Way one:** a librarian. For each question, the librarian brings you the three pages that fit best.

**Way two:** a cheat sheet. One page with everything important. You read it before every question.

#### 1.2 · Two names

**RAG:** retrieval-augmented generation.

**CAG:** cache-augmented generation.

**Retrieval:** fetching something. **Augmented:** something is added. **Generation:** the model writes the answer.

A **cache** is a place where we keep something ready.

#### 1.3 · One sentence for today

<details>
<summary>Click to show</summary>

**RAG and CAG do the same thing: they put knowledge into the prompt. RAG chooses the text for each question. CAG puts all of it in, every time.**

</details>

#### 1.4 · Why do we need this?

**1.** Training data ends at some date.

**2.** The model can make things up.

**3.** It may not know your special field.

---

**Context window:** how much text a model can read at one time.

<div style="height:44px"></div>

## Part 2 of 10 · How the librarian finds the right pages

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**How does the librarian know which pages fit?**

#### 2.1 · The steps of RAG

```
1. The question
2. Turn the question into numbers
3. Search for texts with similar numbers
4. Take the best few texts
5. Put the question and these texts together
6. The model writes the answer
7. The final answer
```

#### 2.2 · A map of meanings

**Embedding model:** turns a text into a long list of numbers.

**Embedding,** or **vector:** this list of numbers. Think of it as an address on a map of meanings.

“dog” and “puppy”: neighbors. “dog” and “tax form”: far apart.

---

**Vector store:** a database that keeps each text together with its address.

**Retriever:** the part that does the search.

<div style="height:44px"></div>

## Part 3 of 10 · Building the library

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What do we prepare before any question comes?**

In [ ]:
# 3.1 · The parts that we need
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

print("✅ Environment setup complete!")

In [ ]:
# 3.2 · Our knowledge base
documents = [
    Document(page_content=(
        "LangGraph is a framework for building stateful, multi-actor applications with "
        "LLMs. It provides a graph-based approach to agent design, allowing developers "
        "to create complex workflows with nodes and edges. LangGraph supports both "
        "synchronous and asynchronous execution, making it suitable for production "
        "applications.")),
    Document(page_content=(
        "Retrieval-Augmented Generation (RAG) is a technique that enhances LLM "
        "responses by retrieving relevant documents from an external knowledge base. "
        "The process involves embedding queries and documents, performing vector "
        "similarity search, and including retrieved context in the LLM prompt. RAG "
        "reduces hallucinations and enables access to up-to-date information.")),
    Document(page_content=(
        "Tools in LangGraph allow agents to interact with external systems. Tools can "
        "be created using the @tool decorator or by extending BaseTool. Agents use "
        "tools by calling them through tool nodes in the graph. Tool execution results "
        "are returned to the agent for further processing.")),
    Document(page_content=(
        "Memory in LangGraph comes in two forms: short-term memory (conversation "
        "history) and long-term memory (persistent storage). Short-term memory is "
        "managed by checkpointers, while long-term memory uses stores with optional "
        "semantic search capabilities. Memory enables agents to maintain context "
        "across conversations.")),
    Document(page_content=(
        "The Model Context Protocol (MCP) is a standardized protocol for connecting AI "
        "agents to external services. MCP servers provide tools and resources that "
        "agents can discover and use. This enables agents to access real-time data, "
        "APIs, and services without custom integration code.")),
]

In [ ]:
# 3.3 · What is in it?
print(f"✅ Created knowledge base with {len(documents)} documents")
for i, doc in enumerate(documents, 1):
    print(f"   {i}. {doc.page_content[:80]}...")

In [ ]:
# 3.4 · Small pieces
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    length_function=len
)

chunks = text_splitter.split_documents(documents)

print(f"✅ Split {len(documents)} documents into {len(chunks)} chunks")
print(f"\nExample chunk:")
print(f"   {chunks[0].page_content[:150]}...")

In [ ]:
# 3.5 · The embedding model
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [ ]:
# 3.6 · Build the library
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="agent_knowledge_base"
)

print("✅ Vector store created with embeddings!")
print(f"   Stored {len(chunks)} document chunks")

<div style="height:44px"></div>

## Part 4 of 10 · The librarian and the RAG chain

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**How does the found text reach the model?**

In [ ]:
# 4.1 · The librarian, alone
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

test_query = "What is LangGraph?"
retrieved_docs = retriever.invoke(test_query)

print(f"✅ Retriever created!")
print(f"\nQuery: '{test_query}'")
print(f"Retrieved {len(retrieved_docs)} documents:\n")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"{i}. {doc.page_content[:200]}...")

#### 4.2 · What did we just see?

<details>
<summary>Click to show</summary>

**The retriever always brings three.**

Even if nothing fits, it brings the three nearest pieces.

So the model must be told what to do when the text does not fit.

</details>

In [ ]:
# 4.3 · The model and the prompt
llm = init_chat_model("gemini-3.5-flash-lite", model_provider="google_genai")

prompt_template = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant that answers questions based on the provided context.
    
Use the following retrieved documents to answer the question. If the context doesn't contain enough information to answer the question, say so.

Context:
{context}"""),
    ("human", "{question}"),
])

In [ ]:
# 4.4 · The chain
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

print("✅ RAG chain created!")

In [ ]:
# 4.5 · Three questions
queries = [
    "What is LangGraph?",
    "How does RAG work?",
    "What are tools in LangGraph?",
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    response = rag_chain.invoke(query)
    print(f"Answer: {response}\n")

#### 4.6 · A rule to remember

<details>
<summary>Click to show</summary>

**RAG is only as good as what it finds.**

The quality of your documents and your search matters as much as the model.

</details>

<div style="height:44px"></div>

## Part 5 of 10 · RAG as a tool for an agent

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What changes when the search is a tool?**

In [ ]:
# 5.1 · A tool that searches
from langchain_core.tools import tool
from agentic_ai.agents.tool_agent import ToolAgent
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_knowledge_base(query: str) -> str:
    """Search the knowledge base for information about AI agents, LangGraph, RAG, tools, and memory.
    
    Use this tool when you need to find information from the knowledge base.
    
    Args:
        query: The search query
        
    Returns:
        Relevant information from the knowledge base
    """
    docs = retriever.invoke(query)

    if docs:
        return "\n\n".join([doc.page_content for doc in docs])
    else:
        return "No relevant information found."

In [ ]:
# 5.2 · The agent
memory = InMemorySaver()
tools = [search_knowledge_base]
agent = ToolAgent(
    llm=llm,
    tools=tools,
    name="RAGAgent",
    checkpointer=memory
)

print("✅ RAG agent created!")
print(f"   Available tools: {[tool.name for tool in tools]}")

In [ ]:
# 5.3 · One question
test_question = "What is RAG and how does it work?"
print(f"Question: {test_question}\n")
print("="*60)

result = agent.run(test_question, thread_id="rag_test_001")
response = result['messages'][-1].text

print(f"Answer:\n{response}")

In [ ]:
# 5.4 · What happened inside?
history = agent.get_conversation_history(thread_id="rag_test_001")

for message in history:
    print(type(message).__name__)
    for call in getattr(message, "tool_calls", []):
        print("   tool_calls:", call["name"], call["args"])
    if message.text:
        print("   " + message.text[:300])

#### 5.5 · What do we take from this?

<details>
<summary>Click to show</summary>

**A tool gives the model text. It does not tell the model to stay with that text.**

For that, you need a prompt.

</details>

<div style="height:44px"></div>

## Part 6 of 10 · The cheat sheet: CAG

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What if the text is simply always in the prompt?**

#### 6.1 · The cheat sheet

**CAG** puts the knowledge into the prompt **before** any question comes.

No search, no embeddings, no vector store.

**The good side:** it is simple, and no search step can bring the wrong pages.

**The bad side:** the whole text goes along with every question, and it must fit into the context window.

In [ ]:
# 6.2 · The same five texts, as one text
cag_knowledge_base = """
# AI Agents Knowledge Base

## LangGraph
LangGraph is a framework for building stateful, multi-actor applications with LLMs. It provides a graph-based approach to agent design, allowing developers to create complex workflows with nodes and edges. LangGraph supports both synchronous and asynchronous execution, making it suitable for production applications.

## Retrieval-Augmented Generation (RAG)
Retrieval-Augmented Generation (RAG) is a technique that enhances LLM responses by retrieving relevant documents from an external knowledge base. The process involves embedding queries and documents, performing vector similarity search, and including retrieved context in the LLM prompt. RAG reduces hallucinations and enables access to up-to-date information.

## Tools in LangGraph
Tools in LangGraph allow agents to interact with external systems. Tools can be created using the @tool decorator or by extending BaseTool. Agents use tools by calling them through tool nodes in the graph. Tool execution results are returned to the agent for further processing.

## Memory in LangGraph
Memory in LangGraph comes in two forms: short-term memory (conversation history) and long-term memory (persistent storage). Short-term memory is managed by checkpointers, while long-term memory uses stores with optional semantic search capabilities. Memory enables agents to maintain context across conversations.

## Model Context Protocol (MCP)
The Model Context Protocol (MCP) is a standardized protocol for connecting AI agents to external services. MCP servers provide tools and resources that agents can discover and use. This enables agents to access real-time data, APIs, and services without custom integration code.
"""

print("✅ Knowledge base prepared for CAG")
print(f"   Knowledge base length: {len(cag_knowledge_base)} characters")

In [ ]:
# 6.3 · A template for the whole text
cag_prompt_template = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant with access to a knowledge base about AI agents.

Use the following knowledge base to answer questions. The knowledge base contains information about LangGraph, RAG, tools, memory, and MCP.

Knowledge Base:
{knowledge_base}

When answering questions:
1. Use information from the knowledge base when relevant
2. If the knowledge base doesn't contain relevant information, say so
3. Be concise and accurate
"""),
    ("human", "{question}"),
])

print("✅ CAG prompt template created!")

In [ ]:
# 6.4 · The chain
cag_chain = (
    {"knowledge_base": lambda x: cag_knowledge_base, "question": RunnablePassthrough()}
    | cag_prompt_template
    | llm
    | StrOutputParser()
)

print("✅ CAG chain created!")

In [ ]:
# 6.5 · The same three questions
queries = [
    "What is LangGraph?",
    "How does RAG work?",
    "What are tools in LangGraph?",
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    response = cag_chain.invoke(query)
    print(f"Answer: {response}\n")

In [ ]:
# 6.6 · A CAG agent
from agentic_ai.agents.simple_agent import SimpleAgent

cag_system_prompt = f"""You are a helpful assistant with access to a knowledge base about AI agents.

Knowledge Base:
{cag_knowledge_base}

When answering questions:
1. Use information from the knowledge base when relevant
2. If the knowledge base doesn't contain relevant information, say so
3. Be concise and accurate
"""

cag_agent = SimpleAgent(
    llm=llm,
    name="CAGAgent",
    system_prompt=cag_system_prompt,
    checkpointer=memory
)

print("✅ CAG agent created!")

In [ ]:
# 6.7 · The same question as for the RAG agent
test_question = "What is RAG and how does it work?"
print(f"Question: {test_question}\n")
print("="*60)

result = cag_agent.run(test_question, thread_id="cag_test_001")
response = result['messages'][-1].text

print(f"Answer:\n{response}")

<div style="height:44px"></div>

## Part 7 of 10 · Which one should you use?

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**How do we compare the two ways?**

#### 7.1 · Two ways, side by side

| | RAG | CAG |
|:--|:--|:--|
| **The text** | chosen for each question | the same for all questions |
| **What it needs** | embeddings, a vector store, a retriever | only a prompt |
| **How much text** | can be very large | must fit into the context window |
| **Good for** | a lot of text, or text that changes often | a small amount of text that stays the same |

In [ ]:
# 7.2 · Three questions again
import time

test_queries = [
    "What is LangGraph?",
    "How does RAG work?",
    "What are tools in LangGraph?",
]

print("Performance Comparison: RAG vs CAG\n")
print("="*60)

In [ ]:
# 7.3 · Measure both
for query in test_queries:
    print(f"\nQuery: {query}")
    print("-"*60)

    start_time = time.time()
    rag_response = rag_chain.invoke(query)
    rag_time = time.time() - start_time

    start_time = time.time()
    cag_response = cag_chain.invoke(query)
    cag_time = time.time() - start_time

    print(f"RAG Time: {rag_time:.3f}s")
    print(f"CAG Time: {cag_time:.3f}s")
    print(f"Speedup: {rag_time/cag_time:.2f}x faster with CAG")

    print(f"\nRAG Response Length: {len(rag_response)} chars")
    print(f"CAG Response Length: {len(cag_response)} chars")

#### 7.4 · What the clock does not show

<details>
<summary>Click to show</summary>

**Tokens that went into the model, in our test:** RAG about 155, CAG about 424.

---

**RAG** costs an extra search, and it can find the wrong pages.

**CAG** costs more tokens, and it has a size limit.

**Neither way is always better.**

</details>

<div style="height:44px"></div>

## Part 8 of 10 · Better searching

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**How can we search better?**

In [ ]:
# 8.1 · Idea one: several questions
def expand_query(query: str) -> list[str]:
    expansions = [
        query,
        f"What is {query}?",
        f"Explain {query}",
        f"Tell me about {query}",
    ]
    return expansions

original_query = "LangGraph"
expanded_queries = expand_query(original_query)

print("Query Expansion Example:")
print(f"Original: {original_query}")
print(f"Expanded: {expanded_queries}")

In [ ]:
# 8.2 · Search with each one
all_docs = []
for eq in expanded_queries:
    docs = retriever.invoke(eq)
    all_docs.extend(docs)

unique_docs = []
seen_content = set()
for doc in all_docs:
    if doc.page_content not in seen_content:
        unique_docs.append(doc)
        seen_content.add(doc.page_content)

print(f"\nRetrieved {len(unique_docs)} unique documents with query expansion")

In [ ]:
# 8.3 · Idea two: check them again
def rerank_documents(query: str, docs: list[Document], top_k: int = 3) -> list[Document]:
    query_words = set(query.lower().split())

    scored_docs = []
    for doc in docs:
        doc_words = set(doc.page_content.lower().split())
        overlap = len(query_words & doc_words)
        scored_docs.append((overlap, doc))

    scored_docs.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scored_docs[:top_k]]

In [ ]:
# 8.4 · Search, then check
query = "LangGraph framework"
docs = retriever.invoke(query)
reranked = rerank_documents(query, docs, top_k=2)

print("Re-ranking Example:")
print(f"Query: {query}")
print(f"\nOriginal retrieval: {len(docs)} documents")
print(f"After re-ranking (top 2): {len(reranked)} documents")
for i, doc in enumerate(reranked, 1):
    print(f"\n{i}. {doc.page_content[:100]}...")

In [ ]:
# 8.5 · Idea three: meaning and exact words
def hybrid_search(query: str, vectorstore, top_k: int = 3) -> list[Document]:
    semantic_docs = vectorstore.similarity_search(query, k=top_k)

    query_words = set(query.lower().split())
    keyword_docs = []
    for doc in chunks:
        doc_words = set(doc.page_content.lower().split())
        if query_words & doc_words:
            keyword_docs.append(doc)

    all_docs = semantic_docs + keyword_docs
    unique_docs = []
    seen_content = set()
    for doc in all_docs:
        if doc.page_content not in seen_content:
            unique_docs.append(doc)
            seen_content.add(doc.page_content)

    return unique_docs[:top_k]

In [ ]:
# 8.6 · Try it
query = "LangGraph"
hybrid_results = hybrid_search(query, vectorstore, top_k=3)

print("Hybrid Search Example:")
print(f"Query: {query}")
print(f"Retrieved {len(hybrid_results)} documents")
for i, doc in enumerate(hybrid_results, 1):
    print(f"\n{i}. {doc.page_content[:100]}...")

<div style="height:44px"></div>

## Part 9 of 10 · Your exercises

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What do you do at home?**

#### 9.1 · First a note, then Exercise 1

Give every vector store its **own `collection_name`**. Stores with the same name in one kernel share their documents.

**Exercise 1:** a RAG system on a topic that you like.

Five documents, a retriever, a chain, and three test questions. **Try one question that your texts cannot answer.**

#### 9.2 · Exercise 2: a cheat sheet

Ten questions and answers, in one text, in the prompt.

Test it with questions that are in the list, and with questions that are not.

#### 9.3 · Exercise 3: compare both

The same knowledge base for both. Measure the time.

Please also count the tokens: remove `StrOutputParser` from the chain, and then `usage_metadata` on the answer gives you the numbers.

#### 9.4 · Exercises 4 and 5, if you have the time

**Exercise 4:** a short cheat sheet in the system prompt, and a RAG tool for everything else.

**Exercise 5:** a re-ranking of your own.

All details are in your own notebook.

<div style="height:44px"></div>

## Part 10 of 10 · Wrap-up

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span></div>

**What should you remember from today?**

#### 10.1 · Five things

**1.** RAG and CAG do the same thing: they put knowledge into the prompt.

**2.** An embedding is an address on a map of meanings, and a retriever finds the nearest neighbors.

**3.** RAG is only as good as what it finds.

**4.** CAG is simple, but every question carries the whole text.

**5.** A tool alone does not keep the model with your text. For that, you need a prompt.

---

**At home:** Exercises 1 and 2.

**Next time, Unit 9:** our agents also work with pictures, sound and video.